# Step 2 — USAD on SKAB (new existing dataset)
Train USAD on SKAB's **anomaly-free** run, test on the 34 labelled experiment files (valve1, valve2, other).
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Takes about 5 minutes.

In [ ]:
!git clone -q https://github.com/manigalati/usad.git
!git clone -q --depth 1 https://github.com/waico/SKAB.git
import sys; sys.path.append('/content/usad')

In [ ]:
import glob, time, numpy as np, pandas as pd, torch
import torch.utils.data as data_utils
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from usad import *
from utils import get_default_device, to_device
device = get_default_device(); print('Device:', device)
torch.manual_seed(0); np.random.seed(0)

WINDOW_SIZE, HIDDEN_SIZE, BATCH_SIZE, N_EPOCHS = 12, 100, 512, 100
D = '/content/SKAB/data'

In [ ]:
# Load SKAB: ';'-separated, 8 sensor channels + 'anomaly' label
def read(f):
    return pd.read_csv(f, sep=';', index_col='datetime', parse_dates=True)

normal = read(f'{D}/anomaly-free/anomaly-free.csv')
FEATURES = [col for col in normal.columns if col not in ('anomaly', 'changepoint')]
test_files = sorted(glob.glob(f'{D}/valve1/*.csv') + glob.glob(f'{D}/valve2/*.csv') + glob.glob(f'{D}/other/*.csv'))

all_test = pd.concat([read(f) for f in test_files])
print('Channels:', len(FEATURES), FEATURES)
print('Normal training rows:', len(normal))
print('Test files:', len(test_files), '| test rows:', len(all_test), '| anomaly %:', round(100 * all_test.anomaly.mean(), 1))

In [ ]:
def make_windows(x, w):
    idx = np.arange(w)[None, :] + np.arange(x.shape[0] - w)[:, None]
    return x[idx]

def point_adjust(pred, label):
    pred = pred.copy(); i, n = 0, len(label)
    while i < n:
        if label[i] == 1:
            j = i
            while j < n and label[j] == 1: j += 1
            if pred[i:j].any(): pred[i:j] = 1
            i = j
        else: i += 1
    return pred

def best_f1(score, label, adjust):
    best = (0, 0, 0, 0)
    for th in np.quantile(score, np.linspace(0.01, 0.9999, 300)):
        pred = (score >= th).astype(int)
        if adjust: pred = point_adjust(pred, label)
        p, r, f, _ = precision_recall_fscore_support(label, pred, average='binary', zero_division=0)
        if f > best[2]: best = (p, r, f, th)
    return best

In [ ]:
# Train ONE USAD model on the anomaly-free run (normal operation only)
scaler = MinMaxScaler().fit(normal[FEATURES].values)
w_train = make_windows(scaler.transform(normal[FEATURES].values), WINDOW_SIZE)
n_feat = len(FEATURES); w_size, z_size = WINDOW_SIZE * n_feat, WINDOW_SIZE * HIDDEN_SIZE
to_t = lambda a: torch.from_numpy(a).float().view(-1, w_size)
split = int(0.8 * len(w_train))
train_loader = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_train[:split])), batch_size=BATCH_SIZE, shuffle=False)
val_loader   = data_utils.DataLoader(data_utils.TensorDataset(to_t(w_train[split:])), batch_size=BATCH_SIZE, shuffle=False)

model = to_device(UsadModel(w_size, z_size), device)
t0 = time.time()
history = training(N_EPOCHS, model, train_loader, val_loader)
print(f'Training time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Score each experiment file
rows, keep = [], {}
for f in test_files:
    df = read(f)
    x = np.clip(scaler.transform(df[FEATURES].values), -5, 5)
    w = make_windows(x, WINDOW_SIZE)
    lab = np.array([1 if df.anomaly.values[i:i+WINDOW_SIZE].sum() > 0 else 0 for i in range(len(w))])
    loader = data_utils.DataLoader(data_utils.TensorDataset(to_t(w)), batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        res = testing(model, loader)
    score = torch.cat([r.flatten() for r in res]).cpu().numpy()
    raw, pa = best_f1(score, lab, False), best_f1(score, lab, True)
    name = f.split('/')[-2] + '/' + f.split('/')[-1]
    keep[name] = (score, lab, pa[3])
    rows.append({'file': name, 'rows': len(df), 'anomaly_%': round(100 * df.anomaly.mean(), 1),
                 'AUC': roc_auc_score(lab, score) if 0 < lab.mean() < 1 else np.nan,
                 'P_raw': raw[0], 'R_raw': raw[1], 'F1_raw': raw[2],
                 'P_PA': pa[0], 'R_PA': pa[1], 'F1_PA': pa[2]})
res_df = pd.DataFrame(rows); res_df.to_csv('skab_results.csv', index=False)
pd.set_option('display.precision', 3); res_df

In [ ]:
# Summary (same format as SMD: F1* = F1 of average P and R)
summary = []
for tag, name in [('PA', 'with point adjustment'), ('raw', 'without point adjustment')]:
    P, R = res_df[f'P_{tag}'].mean(), res_df[f'R_{tag}'].mean()
    summary.append({'setting': name, 'Precision': P, 'Recall': R,
                    'F1 (mean of files)': res_df[f'F1_{tag}'].mean(), 'F1*': 2*P*R/(P+R)})
summary = pd.DataFrame(summary); summary.to_csv('skab_summary.csv', index=False)
print('Mean AUC:', round(res_df.AUC.mean(), 3)); summary

In [ ]:
# Video figure: one valve experiment
name = res_df.sort_values('F1_PA', ascending=False).file.iloc[len(res_df)//2]   # a typical (median) file
score, lab, th = keep[name]
plt.figure(figsize=(12, 4))
plt.semilogy(score, lw=.8, label='USAD anomaly score (log scale)')
plt.fill_between(range(len(lab)), score.min(), score.max(), where=lab == 1, color='red', alpha=.2, label='true anomaly')
plt.axhline(th, color='k', ls='--', lw=1, label='threshold')
plt.xlabel('time window (1 s)'); plt.ylabel('score'); plt.title(f'USAD on SKAB {name}')
plt.legend(loc='upper left'); plt.tight_layout(); plt.savefig('fig_skab_scores.png', dpi=150); plt.show()

In [ ]:
from google.colab import files
for f in ['skab_results.csv', 'skab_summary.csv', 'fig_skab_scores.png']:
    files.download(f)